# Chapter 5: XGBoost Unveiled

While **Iris** dataset is generally used for **classification** problems, the **Diabetes** dataset is used for **regression** problems

In [2]:
# Importing necessary libraries

# pandas is a popular library for working with tabular data (similar to Excel or SQL tables)
import pandas as pd

# numpy is a library for numerical computing in Python.
# It provides support for large multi-dimensional arrays and many mathematical functions.
import numpy as np

# sklearn (short for scikit-learn) is one of the most popular machine learning libraries in Python.
# It provides simple and efficient tools for data mining and data analysis.
from sklearn import datasets

# The Iris dataset

In [3]:
# Loading the famous Iris dataset from sklearn's built-in datasets
# The Iris dataset is a small dataset used frequently for testing machine learning algorithms.
# It contains measurements of 150 iris flowers from 3 different species (setosa, versicolor, virginica).
# Each flower is described by 4 features: sepal length, sepal width, petal length, and petal width.
iris = datasets.load_iris()

---

### Iris Dataset:

👉 `datasets.load_iris()` loads the dataset and returns it as a *Bunch object* — a dictionary-like object with the following useful fields:

* `iris.data`: the features (measurements of the flowers)
* `iris.target`: the labels (which species the flower belongs to)
* `iris.feature_names`: names of the features
* `iris.target_names`: names of the species

---


### Inspecting Iris dataset

see **5_1_Inspecting_Iris_dataset.ipynb**

## Convert the iris dataset into a pandas DataFrame for easier exploration

In [4]:
# np.c_[] is a convenient NumPy function that concatenates arrays *column-wise*.
# In this case, we are concatenating:
#   - iris['data']  → the feature matrix (150 rows x 4 columns)
#   - iris['target'] → the target labels (150 rows x 1 column), reshaped as one column
# Result: a new array with 150 rows and 5 columns (4 features + 1 target)

# pd.DataFrame() is used to create a DataFrame from this array.

# columns= iris['feature_names'] + ['target']
#   - iris['feature_names'] gives us the column names for the 4 features.
#   - We add ['target'] to name the 5th column.

df = pd.DataFrame(
    data = np.c_[iris['data'], iris['target']],  # Combine features and target column-wise
    columns = iris['feature_names'] + ['target'] # Provide column names
)

In [5]:
# View the first 5 rows of the DataFrame to verify
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0.0
1,4.9,3.0,1.4,0.2,0.0
2,4.7,3.2,1.3,0.2,0.0
3,4.6,3.1,1.5,0.2,0.0
4,5.0,3.6,1.4,0.2,0.0


---

***🤔 My SIDEBAR:***

---

***In Machine Learning, we generally split data into two parts:***

| Term                                                                | Meaning                                                                         | Example (Iris dataset)                                                   |
| ------------------------------------------------------------------- | ------------------------------------------------------------------------------- | ------------------------------------------------------------------------ |
| **Predictor columns** (also called *features* or *input variables*) | These are the columns you use to make predictions — the "inputs" to your model. | sepal length (cm), sepal width (cm), petal length (cm), petal width (cm) |
| **Target column** (also called *label* or *output variable*)        | This is the column you are trying to predict — the "output" of your model.      | target (encoded as 0,1,2 — representing species)                         |

---

**In our case:**

```python
df = pd.DataFrame(data= np.c_[iris['data'], iris['target']], columns= iris['feature_names'] + ['target'])
```

* `iris['data']` → contains the **predictor columns** = feature columns

  * sepal length (cm)
  * sepal width (cm)
  * petal length (cm)
  * petal width (cm)

* `iris['target']` → is the **target column** (label):

  * 0 → setosa
  * 1 → versicolor
  * 2 → virginica

---

***🤔 END My SIDEBAR***

---

# Building XGBoost classification templates

### Importing XGBoost Classifier model

In [6]:
# XGBoost (Extreme Gradient Boosting) is a powerful machine learning algorithm based on decision trees.
# It is very efficient and often provides high accuracy in classification tasks.
# XGBClassifier is used when your target is categorical (classification problem).
from xgboost import XGBClassifier

# Importing train_test_split function
# This function splits your dataset into two parts:
#  - Training set (used to train the model)
#  - Test set (used to evaluate the model's performance on unseen data)
from sklearn.model_selection import train_test_split

# Importing accuracy_score metric
# This is a common evaluation metric for classification problems.
# It tells you what fraction of the test samples were correctly classified by the model.
# Accuracy ranges from 0 to 1 → higher is better.
from sklearn.metrics import accuracy_score

### Split the Iris dataset into training and testing sets

In [7]:
# train_test_split() will randomly split your data into:
#   - X_train → Features used to train the model
#   - X_test → Features used to test the model (evaluate performance)
#   - y_train → Target values used to train the model
#   - y_test → Target values used to test the model

# iris['data'] → The feature matrix (all 150 rows of 4 feature columns)
# iris['target'] → The target labels (species of each flower)

# random_state=2 → Setting a random seed for reproducibility.
# This ensures that every time you run this code, you will get the *same split* of data.

X_train, X_test, y_train, y_test = train_test_split(
    iris['data'],   # Features (inputs)
    iris['target'], # Target (outputs / labels)
    random_state=2  # Random seed for reproducibility
)


### Use XGBClassifier to Train, Predict and Evaluate

In [8]:
# STEP 1: Initialize the XGBoost Classifier model

xgb = XGBClassifier(
    booster='gbtree',          # Use tree-based boosting (standard choice for tabular data)
    objective='multi:softprob',# Multi-class classification (output = probability distribution over classes)
    learning_rate=0.1,         # Learning rate (step size shrinkage) — lower values improve stability but require more trees
    n_estimators=100,          # Number of trees (boosting rounds) — 100 is a typical starting value
    random_state=2,            # For reproducibility (same results every time you run the code)
    n_jobs=-1                  # Use all available CPU cores (parallel training for faster performance)
)

# STEP 2: Train (fit) the model on the training set
# The model "learns" from the (X_train, y_train) pairs
xgb.fit(X_train, y_train)

# STEP 3: Use the trained model to make predictions on the test set
# This will output the predicted class labels for each sample in X_test
y_pred = xgb.predict(X_test)

# STEP 4: Evaluate model performance using accuracy_score
# accuracy_score = (number of correct predictions) / (total number of predictions)
score = accuracy_score(y_pred, y_test)

# STEP 5: Print the accuracy score
print('Score: ' + str(score))

Score: 0.9736842105263158


**Note**: While **accuracy_score** is standard, other scoring methods, such as **auc** (Area Under Curve) can also be tried

---

**<u>Describing some of the Hyperparameters used</u>**

a) **booster='gbtree'**: The booster is the base learner. It's the machine learning model that is constructed during every round of boosting. You may have guessed that 'gbtree' stands for gradient boosted tree, the XGBoost default base learner. It's uncommon but possible to work with other base learners

b) **objective='multi:softprob'**: Standard options for the objective can be viewed in the XGBoost official documentation, https://xgboost.readthedocs.io/en/latest/parameter.html, under Learning Task Parameters. The multi:softprob objective is a standard alternative to binary:logistic when the dataset includes multiple classes. It computes the probabilities of classification and chooses the highest one. If not explicitly stated, XGBoost will often find the right objective for you.

c) **max_depth=6**: The max_depth of a tree determines the number of branches each tree has. It's one of the most important hyperparameters in making balanced predictions. XGBoost uses a default of 6, unlike random forests, which don't provide a value unless explicitly programmed.

d) **learning_rate=0.1**: Within XGBoost, this hyperparameter is often referred to as eta. This hyperparameter limits the variance by reducing the weight of each tree to the given percentage.

e) **n_estimators=100**: Popular among ensemble methods, n_estimators is the number of boosted trees in the model. Increasing this number while decreasing learning_rate can lead to more robust results.

---

# The Diabetes dataset

### Import necessary tools

In [9]:
# Import necessary tools
from sklearn.model_selection import cross_val_score  # For cross-validation
from xgboost import XGBRegressor                     # XGBoost model for regression

### Load the diabetes dataset (built-in sklearn dataset)

In [10]:
# return_X_y=True → returns features (X) and target (y) as NumPy arrays
X, y = datasets.load_diabetes(return_X_y=True)

### STEP 1: Initialize the XGBoost Regressor model

In [11]:
xgb = XGBRegressor(
    booster='gbtree',                # Use tree-based boosting (standard choice for regression tasks)
    objective='reg:squarederror',    # Regression with squared error (mean squared error loss)
    learning_rate=0.1,               # Learning rate — smaller = more stable but slower to converge
    n_estimators=100,                # Number of boosting rounds (trees)
    random_state=2,                  # For reproducibility
    n_jobs=-1                        # Use all CPU cores for faster training
)

### STEP 2: Perform cross-validation

In [12]:
# cross_val_score will:
# - Split the data into 'cv' folds (5 in this case)
# - For each fold:
#     - Train the model on 4 folds
#     - Test the model on the remaining fold
# - Repeat this 5 times, so every fold is used for testing once
# - Return an array of scores (one per fold)

# 'scoring' → specifies the metric:
# - 'neg_mean_squared_error' = negative MSE (because sklearn always maximizes the score → negative sign is convention)
scores = cross_val_score(
    xgb, X, y,
    scoring='neg_mean_squared_error',
    cv=5
)

### STEP 3: Compute Root Mean Squared Error (RMSE)

In [13]:
# RMSE = sqrt(MSE), a more interpretable metric (same units as target variable)
# Since scores are negative (due to sklearn convention), take -scores and then sqrt
rmse = np.sqrt(-scores)

### STEP 4: Display RMSE for each fold

In [14]:
print('RMSE:', np.round(rmse, 3))
# Example RMSE: [63.033 59.689 64.538 63.699 64.661]

RMSE: [59.397 60.322 69.036 63.211 66.953]


### STEP 5: Display mean RMSE across all folds (gives overall model performance)

In [15]:
print('RMSE mean: %0.3f' % (rmse.mean()))
# Example RMSE mean: 63.124

RMSE mean: 63.784


In [16]:
pd.DataFrame(y, columns=['disease_progression']).describe()

,disease_progression
count,442.000000
mean,152.133484
std,77.093005
min,25.000000
25%,87.000000
50%,140.500000
75%,211.500000
max,346.000000


---

### ✅ What this does:

* `y` is a NumPy array of target values (disease progression scores).
* Wrapping it in `pd.DataFrame(y)` converts it into a DataFrame so you can use `describe()`.

The `.describe()` function provides **summary statistics**, including:

| Metric    | Meaning                                    |
| --------- | ------------------------------------------ |
| **count** | Number of rows (samples)                   |
| **mean**  | Average value                              |
| **std**   | Standard deviation (spread of data)        |
| **min**   | Minimum value                              |
| **25%**   | First quartile (25% of data is below this) |
| **50%**   | Median (middle value)                      |
| **75%**   | Third quartile (75% of data is below this) |
| **max**   | Maximum value                              |

---

### 📊 Example output:

```text
               0
count  442.000000
mean   152.133484
std     77.093005
min     25.000000
25%     87.000000
50%    140.500000
75%    211.500000
max    346.000000
```

This tells us:

* There are 442 samples
* The average disease progression score is about **152**
* Scores range from **25** to **346**
* 75% of patients had scores **≤ 211.5**

---

## RMSE Ratio

A score of 63.124 is less than 1 standard deviation, a respectable result.

In [18]:
# Compute std(y)
y_std = np.std(y)

# Compute mean RMSE
rmse_mean = rmse.mean()

# Compute RMSE / std(y) ratio
rmse_ratio = rmse_mean / y_std

# Display result
print(f'RMSE / std(y) ratio: {rmse_ratio:.2f}')

RMSE / std(y) ratio: 0.83


see sidebar: **5_2_RMSE_Ratio.ipynb**